In [ ]:
# Inputs: DataFrame with a row for each fire point that needs feature data

# Outputs: DataFrame of GEE feature data

# Purpose: Get feature data from GEE

In [ ]:
import ee
import numpy as np
import pandas as pd
import geopandas
from pyproj import Proj, Transformer
import os
import time
from itertools import batched
from pathlib import Path

In [ ]:
 ee.Authenticate(auth_mode='localhost', force=True)

In [ ]:
ee.Initialize(project='geewildfires')

In [ ]:
def coordinates_from_geodataframe(df):
    transformer = Transformer.from_crs(df['geometry'].crs, "EPSG:4326", always_xy=True)
    df["lon1"], df["lat1"] = transformer.transform(df["geometry"].x, df["geometry"].y)
    df = df[["FIRE_ID", "IG_DATE", "Occurrence", "point_orde", "lon1", "lat1"]]
    return df

def extract_features_from_dataframe_points(df):
    features=[]
    for index, row in df.iterrows():
        point_geometry = ee.Geometry.Point(row['lon1'], row['lat1'])
        point_properties = {'fire_id': row["FIRE_ID"], 'date': str(row["IG_DATE"]), 'occur_id':  str(row["Occurrence"]), "point_id": str(row["point_orde"])}
        point_feature = ee.Feature(point_geometry, point_properties)
        features.append(point_feature)
        
    feature_collection = ee.FeatureCollection(features)
    return feature_collection

def year_extract_features_from_dataframe_points(df):
    features=[]
    for index, row in df.iterrows():
        point_geometry = ee.Geometry.Point(row['lon1'], row['lat1'])
        point_properties = {'fire_id': row["FIRE_ID"], 'date': str(row["IG_DATE"]), 'occur_id':  str(row["Occurrence"]), "point_id": str(row["point_orde"])}
        point_feature = ee.Feature(point_geometry, point_properties)
        features.append(point_feature)
        
    feature_collection = ee.FeatureCollection(features)
    return feature_collection

def get_terrain_collection(collection_name):
    terrain = ee.ImageCollection(collection_name)
    return terrain

def year_get_image_for_date(collection_name, date_string, feature):
    date = str(date_string[:10])
    date_counter = 730
    image_found = False
    while not image_found:
        date_start = str((pd.Timestamp(date) - pd.Timedelta(days=date_counter)).date())
        selected_collection = ee.ImageCollection(collection_name).filterDate(date_start, date)
        filtered_test = selected_collection.filterBounds(feature.geometry())
        if filtered_test.limit(1).size().getInfo():
            image_found = True
        elif date_counter > 2000:
            return None
        else:
            date_counter += 365
    return selected_collection, date_counter

def get_image_for_date(collection_name, date_string, feature):
    date = str(date_string[:10])
    date_counter = 17
    image_found = False
    while not image_found:
        date_start = str((pd.Timestamp(date) - pd.Timedelta(days=date_counter)).date())
        selected_collection = ee.ImageCollection(collection_name).filterDate(date_start, date)
        filtered_test = selected_collection.filterBounds(feature.geometry())
        if date_counter > 180:
            return None
        elif filtered_test is None:
            date_counter += 17
        else:
            image_found = True
    return selected_collection, date_counter

def get_image_for_date_final(collection_name, date_string, feature):
    date = str(date_string[:10])
    date_counter = 20
    image_found = False
    while not image_found:
        date_start = str((pd.Timestamp(date) - pd.Timedelta(days=date_counter)).date())
        selected_collection = ee.ImageCollection(collection_name).filterDate(date_start, date)
        filtered_test = selected_collection.filterBounds(feature.geometry())
        if filtered_test.size().getInfo() > 0:
            image_found = True
        elif date_counter > 180:
            return None
        else:
            date_counter += 20
    return selected_collection, date_counter

def merge_feature_collection_list(feature_collection_list):
    result_feature_collection = feature_collection_list[0]
    for feature_collection in feature_collection_list[1:]:
        result_feature_collection = result_feature_collection.merge(feature_collection)
    return result_feature_collection

def set_target_projection(target_projection):
    if isinstance(target_projection, ee.Projection):
        target_proj = target_projection
    else:
        target_proj = ee.Projection(f'EPSG:{target_projection}')
    return target_proj

def raster_reduce(image, feature, scale=30):
    feature_sample = image.reduceRegions(
        collection = feature,
        reducer=ee.Reducer.first(),
        scale = scale,
    )
    return feature_sample

def terrain_reduce(image, feature, scale=90):
    terrain = ee.Terrain.products(image)
    feature_sample = terrain.reduceRegions(
        collection = feature,
        reducer=ee.Reducer.first(),
        scale = scale,
    )
    return feature_sample

def apply_raster_extraction(collection_name, measures, feature, target_projection):
    target_proj = set_target_projection(target_projection)
    filtered = collection_name.filterBounds(feature.geometry())
    results = filtered.select(measures).filterBounds(feature.geometry()).mosaic().setDefaultProjection(target_proj)
    return results

def export_to_drive(tasks, raster_sampled, description, i, request_id):
    task = ee.batch.Export.table.toDrive(
        collection=raster_sampled,
        description=f'{description}_{i}_{request_id}',
        folder='wildfire_lines',
        fileNamePrefix=f'{description}_{i}_{request_id}',
        fileFormat='CSV'
    )
    task.start()
    tasks.append(task)

In [ ]:
def get_dated_point_data(df, collection, measure_list, description, dated=True, request_id=1):
    fires_df_coords = coordinates_from_geodataframe(df)
    extracted_feature_list = []
    tasks = []
    batch_all_features = []
    i = 0
    request_count = 0
    request_old = 0
        
    date_batches = [(date, date_group) for date, date_group in fires_df_coords.groupby('IG_DATE')]
    total_batches = len(date_batches)

    for date, date_group in date_batches:
        i += 1
        try:
            point_feature_collection = extract_features_from_dataframe_points(date_group)
            date_str = date

            result_image = get_image_for_date_final(collection, date_str, point_feature_collection)
            if result_image is None:
                continue
            test_collection, days_back = result_image

            raster_extracted = apply_raster_extraction(test_collection, measure_list, point_feature_collection, 4326)
            raster_reduced = raster_reduce(raster_extracted, point_feature_collection, scale=30)

            batch_all_features.append(raster_reduced)
        
        except:
            # future improvement: log failed FIRE_IDs here
            continue
    
    print(f"{len(batch_all_features)} feature collections out of {total_batches} date batches.")
    batch_check = 0

    for feature_collection_group in batched(batch_all_features, 2):
        request_id += 1
        batch_check += 1
        request_old = request_count + 1
        request_count += 2
        raster_batched = merge_feature_collection_list(feature_collection_group)
        print(f"Getting dates from {request_old} to {request_count}")
        export_to_drive(tasks, raster_batched, description, i, request_id)
        if batch_check > 2000:
            time.sleep(300)
            print("Waiting for queue to clear")
            batch_check = 0
        

In [ ]:
def get_fixed_point_data(df, collection, measure_list, description, dated=False, request_id=1):
    fires_df_coords = coordinates_from_geodataframe(df)
    extracted_feature_list = []
    tasks = []
    batch_all_features = []
    i = 0
    request_count = 0
    request_old = 0
    
    fires_df_coords['year'] = pd.to_datetime(fires_df_coords['IG_DATE']).dt.year
    fires_df_coords['month'] = pd.to_datetime(fires_df_coords['IG_DATE']).dt.month
    fires_df_coords['year_month'] = fires_df_coords['year'].astype(str) + '_' + fires_df_coords['month'].astype(str)
    
    date_batches = [(date, date_group) for date, date_group in fires_df_coords.groupby('year_month')]
    total_batches = len(date_batches)
    
    for date, date_group in date_batches:
        i += 1
        try:
            point_feature_collection = extract_features_from_dataframe_points(date_group)
            test_collection = get_terrain_collection(collection)
            raster_extracted = apply_raster_extraction(test_collection, measure_list, point_feature_collection, test_collection.first().projection())
            raster_reduced = terrain_reduce(raster_extracted, point_feature_collection, scale=90)

            batch_all_features.append(raster_reduced)
            
        except:
            # future improvement: log failed FIRE_IDs here
            continue
    
    print(f"{len(batch_all_features)} feature collections out of {total_batches} batches.")
    batch_check = 0

    for feature_collection_group in batched(batch_all_features, 1):
        request_id += 1
        batch_check += 1
        request_old = request_count + 1
        request_count += 1
        raster_batched = merge_feature_collection_list(feature_collection_group)
        print(f"Getting dates from {request_old} to {request_count}")
        export_to_drive(tasks, raster_batched, description, i, request_id)
        if batch_check > 1000:
            time.sleep(300)
            print("Waiting for queue to clear")
            batch_check = 0

In [ ]:
def get_year_point_data(df, collection, measure_list, description, dated=True, request_id=1):
    fires_df_coords = coordinates_from_geodataframe(df)
    extracted_feature_list = []
    tasks = []
    batch_all_features = []
    i = 0
    request_count = 0
    request_old = 0

    fires_df_coords['year'] = pd.to_datetime(fires_df_coords['IG_DATE']).dt.year
    fires_df_coords['month'] = pd.to_datetime(fires_df_coords['IG_DATE']).dt.month
    fires_df_coords['month'] = fires_df_coords['month'].astype(str).str.zfill(2)
    fires_df_coords['year_month'] = fires_df_coords['year'].astype(str) + '-' + fires_df_coords['month'].astype(str)
    
    date_batches = [(date, date_group) for date, date_group in fires_df_coords.groupby('year_month')]
    total_batches = len(date_batches)

    for date, date_group in date_batches:
        i += 1
        date_pop = str(date)
        try:
            point_feature_collection = year_extract_features_from_dataframe_points(date_group)
            date_str = set_five_year_date_pop(date_pop)

            result_image = year_get_image_for_date(collection, date_str, point_feature_collection)
            if result_image is None:
                continue
            test_collection, days_back = result_image
            raster_extracted = apply_raster_extraction(test_collection, measure_list, point_feature_collection, 4326)
            raster_reduced = raster_reduce(raster_extracted, point_feature_collection, scale=30)
            size = raster_reduced.size().getInfo()

            batch_all_features.append(raster_reduced)
        
        except:
            # future improvement: log failed FIRE_IDs here
            continue
    
    print(f" {len(batch_all_features)} feature collections out of {total_batches} batches.")
    batch_check = 0

    for feature_collection_group in batched(batch_all_features, 1):
        request_id += 1
        batch_check += 1
        request_old = request_count + 1
        request_count += 1
        raster_batched = merge_feature_collection_list(feature_collection_group)
        print(f"Getting dates from {request_old} to {request_count}")
        export_to_drive(tasks, raster_batched, description, i, request_id)
        if batch_check > 2000:
            time.sleep(300)
            print("Waiting for queue to clear")
            batch_check = 0

In [ ]:
landsat7_collection = "LANDSAT/LE07/C02/T1_L2"
landsat7_measures = ['SR_B2','SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']
landsat5_collection = "LANDSAT/LT05/C02/T1_L2"
landsat5_measures = ['SR_B2','SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']
landsat8_collection = "LANDSAT/LC08/C02/T1_L2"
landsat8_measures = ['SR_B2','SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']

elevation_collection = 'COPERNICUS/DEM/GLO30'
elevation_measures = ['DEM']

weather_metrics = ['tmmx', 'tmmn', 'pr', 'rmax', 'rmin', 'vs', 'th', 'erc', 'bi', 'fm100', 'fm1000', 'eto', 'vpd']
weather_collection = 'IDAHO_EPSCOR/GRIDMET'

pop_density_collection = 'CIESIN/GPWv411/GPW_Population_Density'
pop_density_metrics = 'population_density'

land_cover_collection = "JRC/GHSL/P2023A/GHS_SMOD_V2-0"
land_cover_metrics = "smod_code"

# remember the notebook needs to be Trusted for the calls to work properly

In [ ]:
def download_landsat_data(df, r_id=1):
    # added a couple of months to each of these, just because program needs to look back for images
    fires_ldf_8 = df[df['IG_DATE'] > '2013-05-17']
    fires_ldf_7 = df.loc[(df['IG_DATE'] > '1999-07-28') & (df['IG_DATE'] < '2013-05-17')]
    fires_ldf_5 = df[df['IG_DATE'] < '1999-07-29']
    fires_df = df
    
    if len(fires_ldf_8) >= 0:
        landsat_8_data = get_dated_point_data(fires_ldf_8, landsat8_collection, ['SR_B2','SR_B3', 'SR_B4', 'SR_B5', 'SR_B7'], "landsat8", request_id=r_id)
    if len(fires_ldf_7) >= 0:
        landsat_7_data = get_dated_point_data(fires_ldf_7, landsat7_collection, ['SR_B2','SR_B3','SR_B4', 'SR_B5', 'SR_B7'], "landsat7", request_id=r_id)
    if len(fires_ldf_5) >= 0:
        landsat_5_data = get_dated_point_data(fires_ldf_5, landsat5_collection, ['SR_B2','SR_B3','SR_B4', 'SR_B5', 'SR_B7'], "landsat5", request_id=r_id)
    
def download_gridmet_data(df, r_id = 1):
    gridmet_data = get_dated_point_data(df, weather_collection, weather_metrics, "weather", request_id=r_id)
    
def download_elevation_data(df, r_id=1):
    terrain_data = get_fixed_point_data(df, elevation_collection, ['DEM'], "elevation", dated=False, request_id=r_id)
    
def download_pop_density_data(df, r_id = 1):
    pop_density_data = get_year_point_data(df, pop_density_collection, pop_density_metrics, "pop_density", request_id=r_id)

def download_landcover_data(df, r_id = 1):
    land_cover_data = get_year_point_data(df, land_cover_collection, land_cover_metrics, "land_cover", request_id=r_id)

In [ ]:
def set_five_year_date(date):
    new_date = '2020-12-31'
    year = pd.to_datetime(date).year
    if year >= 2015:
        new_date = '2015-12-31'
    elif year >= 2010:
        new_date = '2010-12-31'
    elif year >= 2005:
        new_date = '2005-12-31'
    elif year >= 2000:
        new_date = '2000-12-31'
    elif year >= 1995:
        new_date = '1995-12-31'
    elif year >= 1990:
        new_date = '1990-12-31'
    elif year >= 1985:
        new_date = '1985-12-31'
    elif year >= 1980:
        new_date = '1980-12-31'
        
    return new_date

def set_five_year_date_pop(date):
    new_date = '2020-12-31'
    year = pd.to_datetime(date).year
    if year >= 2015:
        new_date = '2015-12-31'
    elif year >= 2010:
        new_date = '2010-12-31'
    elif year >= 2005:
        new_date = '2005-12-31'
    elif year >= 1970:
        new_date = '2000-12-31'
        
    return new_date

In [ ]:
def load_gee_from_folder(folder, data_name):
    import os
    dfs = []
    for filename in os.listdir(folder):
        if data_name in filename:
            file_r = pd.read_csv(os.path.join(folder, filename), engine='python')
            dfs.append(file_r)
    all_df = pd.concat(dfs, axis=0, ignore_index=True)
    return all_df

In [ ]:
fires_ldf = geopandas.read_file(r"C:\Users\jezkn\OneDrive\Documents\Birkbeck\Work\MSc Project\Wildfire Project\Outputs\FINAL_PartTwoOutput3.shp")
fires_ldf = fires_ldf
print(len(fires_ldf))

In [ ]:
fires_ldf_trunc = fires_ldf.to_crs(5070)
fires_ldf_trunc

In [ ]:
download_elevation_data(fires_ldf_trunc, r_id=1)

In [ ]:
download_gridmet_data(fires_ldf_trunc, r_id=1)

In [ ]:
download_landsat_data(fires_ldf_trunc, r_id=1)

In [ ]:
download_pop_density_data(fires_ldf_trunc, r_id=1)

In [ ]:
download_landcover_data(fires_ldf_trunc, r_id=1)

In [ ]:
# reminder: run each of the downloads and wait for them to complete before continuing

In [ ]:
import gc
gc.collect()

In [ ]:
# reminder: download everything from Google Drive into the local folder

In [ ]:
folder = r"C:\Users\jezkn\OneDrive\Documents\Birkbeck\Work\MSc Project\Wildfire Project\GEE_Downloads\20260827_wildfire\wildfire_lines"

topology_df = load_gee_from_folder(folder, 'elevation')
weather_df = load_gee_from_folder(folder, 'weather')
landsat8_df = load_gee_from_folder(folder, 'landsat8')
landsat7_df = load_gee_from_folder(folder, 'landsat7')
landsat5_df = load_gee_from_folder(folder, 'landsat5')
pop_density_df = load_gee_from_folder(folder, 'pop_density')
land_cover_df = load_gee_from_folder(folder, 'land_cover')
topology_df.head()

In [ ]:
topology_df = topology_df.drop(['system:index'], axis=1)
weather_df = weather_df.drop(['system:index'], axis=1)
landsat8_df = landsat8_df.drop(['system:index'], axis=1)
landsat7_df = landsat7_df.drop(['system:index'], axis=1)
landsat5_df = landsat5_df.drop(['system:index'], axis=1)
pop_density_df= pop_density_df.drop(['system:index', 'date'], axis=1)
land_cover_df = land_cover_df.drop(['system:index', 'date'], axis=1)

In [ ]:
concat_df = pd.concat([landsat8_df, landsat7_df, landsat5_df])

In [ ]:
len(concat_df)

In [ ]:
concat_df.info(verbose=True, show_counts=True)

In [ ]:
len(pop_density_df)

In [ ]:
pop_density_df['pop_density'] = pop_density_df['first']
pop_density_df.drop(['first', '.geo'], axis=1, inplace=True)

pop_density_df.columns
len(pop_density_df)

In [ ]:
pop_density_df.drop_duplicates(subset=['fire_id', 'occur_id', 'point_id'], inplace=True)
len(pop_density_df)

In [ ]:
pop_density_df.info(verbose=True, show_counts=True)

In [ ]:
print(land_cover_df.columns)
len(land_cover_df)

In [ ]:
land_cover_df['u_class'] = land_cover_df['first']
land_cover_df.drop(['first', '.geo'], axis=1, inplace=True)

In [ ]:
land_cover_df.drop_duplicates(subset=['fire_id', 'occur_id', 'point_id'], inplace=True)
len(land_cover_df)

In [ ]:
final_df = concat_df.merge(topology_df, how='left', on=['fire_id', 'occur_id', 'point_id'])
len(final_df)

In [ ]:
final_df = final_df.merge(weather_df, how='left', on=['fire_id', 'occur_id', 'point_id'])
len(final_df)

In [ ]:
final_df = final_df.merge(pop_density_df, how='left', on=['fire_id', 'occur_id', 'point_id'])
len(final_df)

In [ ]:
final_df = final_df.merge(land_cover_df, how='left', on=['fire_id', 'occur_id', 'point_id'])
len(final_df)

In [ ]:
final_df.columns

In [ ]:
len(final_df)

In [ ]:
final_df.info(verbose=True, show_counts=True)

In [ ]:
final_df['geometry'] = final_df['.geo'].fillna(final_df[".geo_x"]).fillna(final_df[".geo_y"])
final_df = final_df.drop(['.geo_x', '.geo_y', '.geo'], axis=1)

In [ ]:
final_df.count()

In [ ]:
final_df

In [ ]:
final_df = final_df.drop(columns=['geometry'])

In [ ]:
final_df.columns

In [ ]:
final_df.to_csv(r"C:\Users\jezkn\OneDrive\Documents\Birkbeck\Work\MSc Project\Wildfire Project\Outputs\FINAL_PartThreeOutput2.csv")